# Chronopharmacology and Molecular Profiling Data from 5-ASA Treated DSS-Induced Colitis in Mice Exploration with `mlcroissant`
This notebook provides a structured guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.pkwe-az16/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.pkwe-az16/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access metadata (as a single object)
metadata = dataset.metadata
print("\nDataset Name: {}\nDescription: {}\nPublished: {}\nKeywords: {}\n".format(
    metadata.name,
    metadata.description,
    getattr(metadata, 'datePublished', 'N/A'),
    getattr(metadata, 'keywords', 'N/A')
))

## 2. Data Overview
Review available record sets, fields, and their unique `@id` identifiers.

The `mlcroissant` library allows us to access the Croissant schema dynamically, listing all available record sets and their fields. Every entity is referenced by its `@id` as required.

In [ ]:
# List all record sets and their IDs
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"- Record Set Name: {rs.name}, @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {getattr(field, 'dataType', 'N/A')})")
    print("---")
# Assign one record set for demonstration if available
if len(record_sets) > 0:
    sample_record_set = record_sets[0]  # Use the first for notebook steps
    sample_record_set_id = sample_record_set.id
else:
    sample_record_set_id = None

## 3. Data Extraction
Load data from selected record sets into pandas DataFrames for further analysis. All loading is referenced using record set and field `@id`s.

In [ ]:
# Collect all record set IDs for batch extraction
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load records, referencing by @id
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        dataframes[record_set_id] = pd.DataFrame()

# Show columns for the sample record set
if sample_record_set_id in dataframes:
    print(f"Columns for record set @id {sample_record_set_id}: {dataframes[sample_record_set_id].columns.tolist()}")
    display(dataframes[sample_record_set_id].head())
else:
    print("No records loaded for the sample record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps with respect to the field `@id`s. These include filtering records, removing outliers, normalizing numeric fields, or grouping by key attributes.

Below, we filter and normalize a numeric field and group by a categorical field.
- Choose numeric and group fields by reviewing the sample record set's fields and their `@id`.

In [ ]:
# Identify numeric fields from sample_record_set
numeric_field_id = None
group_field_id = None

if sample_record_set_id:
    for field in sample_record_set.fields:
        if getattr(field, 'dataType', '').startswith('schema:Float') or getattr(field, 'dataType', '').startswith('schema:Integer'):
            numeric_field_id = field.id
        elif getattr(field, 'dataType', '').startswith('schema:Text'):
            group_field_id = field.id
    print(f"Selected numeric field @id: {numeric_field_id}, group field @id: {group_field_id}")

# EDA if fields exist
if numeric_field_id and sample_record_set_id in dataframes and not dataframes[sample_record_set_id].empty:
    df = dataframes[sample_record_set_id]
    # Remove rows with missing values for numeric field
    df = df[df[numeric_field_id].notnull()]
    # Filter by threshold
    threshold = 10
    filtered_df = df[df[numeric_field_id].astype(float) > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
    ) / filtered_df[numeric_field_id].astype(float).std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by categorical field
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
else:
    print("Could not identify appropriate fields or records for EDA.")

## 5. Visualization
Visualize distributions or relationships using matplotlib/seaborn. All field references are given by their `@id`.

Below, we plot the distribution of the normalized numeric field and create a boxplot grouped by the categorical field, if available.

In [ ]:
# Visualization
if numeric_field_id and sample_record_set_id in dataframes and not dataframes[sample_record_set_id].empty:
    df = dataframes[sample_record_set_id]
    if df[numeric_field_id].dtype != object:
        values = df[numeric_field_id].dropna().astype(float)
    else:
        values = pd.to_numeric(df[numeric_field_id].dropna(), errors='coerce')
    plt.figure(figsize=(8, 4))
    sns.histplot(values, bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If normalized field exists
    norm_col = f"{numeric_field_id}_normalized"
    if norm_col in df.columns:
        plt.figure(figsize=(8, 4))
        sns.histplot(df[norm_col].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {norm_col}")
        plt.xlabel(norm_col)
        plt.ylabel("Frequency")
        plt.show()

    # Boxplot grouped by categorical field
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=df[group_field_id], y=values)
        plt.title(f"Boxplot of {numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("Visualization not available due to missing or incompatible fields.")

## 6. Conclusion
In this notebook, we explored the FAIR^2 dataset using `mlcroissant`, referencing all record sets, fields, and columns by their `@id`. After loading and overviewing the data structure, we extracted tabular data, performed EDA, and visualized selected attributes. This approach ensures reproducibility and structural consistency, facilitating further pharmacological and chronobiological analysis.

For more advanced analyses, refer to the dataset documentation and protocol fields provided in the metadata.